# 02 - Algoritmo Random Forest en Clasificación

## Curso de Machine Learning: Modelos de Clasificación

El algoritmo **Random Forest (Bosques Aleatorios)**, propuesto por Leo Breiman en 2001, representa una de las mayores innovaciones del aprendizaje automático. Mejora radicalmente a Bagging al incorporar un segundo mecanismo de aleatoriedad: la **selección aleatoria de subconjuntos de características (Feature Subsampling)** en cada nodo de cada árbol.

---

### Contenidos
1. Limitaciones de Bagging Simple y el Problema de la Correlación entre Árboles
2. Algoritmo Random Forest: Doble Aleatoriedad (Muestras y Características)
3. Regla Heurística de Características: $m = \sqrt{D}$
4. Hiperparámetros Fundamentales (`n_estimators`, `max_features`, `class_weight`)
5. Curva de Estabilización OOB vs. Número de Árboles
6. Interpretabilidad: MDI (Gini Importance) vs. Permutation Importance
7. Visualización 2D de la Frontera de Decisión Suavizada
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import DecisionBoundaryDisplay, permutation_importance
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. El Problema de la Correlación entre Árboles en Bagging

Supongamos un conjunto de datos donde existe una variable fuertemente predictora junto a varias variables moderadamente informativas:
- En Bagging convencional, **la mayoría de los árboles elegirán la variable más fuerte para la división en el nodo raíz**.
- En consecuencia, todos los árboles se volverán estructuralmente muy parecidos y sus predicciones estarán altamente correlacionadas.
- La varianza del promedio de $B$ variables aleatorias con varianza $\sigma^2$ y correlación par a par $\rho$ es:

$$\mathbb{V}ar(\bar{X}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

A medida que $B \to \infty$, el segundo término tiende a cero, pero **el primer término $\rho \sigma^2$ persiste**. Para reducir la varianza al mínimo, es imperativo reducir $\rho$.

### La Solución de Random Forest:
En cada división de cada árbol, se eligen al azar únicamente $m$ características candidatas de las $D$ totales (típicamente $m = \lfloor \sqrt{D} \rfloor$). Esto **fuerza a los árboles a explorar variables secundarias**, descorrelacionándolos drásticamente.


In [ ]:
# Demostracion: Curva de convergencia del error OOB vs Numero de Estimadores
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

n_estimators_list = [10, 25, 50, 75, 100, 150, 200, 300]
oob_errors = []
test_errors = []

for n in n_estimators_list:
    rf = RandomForestClassifier(n_estimators=n, max_features='sqrt', oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    oob_errors.append(1.0 - rf.oob_score_)
    test_errors.append(1.0 - rf.score(X_test, y_test))

plt.figure(figsize=(10, 4.8))
plt.plot(n_estimators_list, oob_errors, 'o-', color='#2b5c8f', lw=2, label='Error OOB (Entrenamiento)')
plt.plot(n_estimators_list, test_errors, 's-', color='#d95f02', lw=2, label='Error en Test')
plt.xlabel('Numero de Arboles (n_estimators)', fontweight='bold')
plt.ylabel('Tasa de Error', fontweight='bold')
plt.title('Estabilizacion del Error en Random Forest: Mas arboles NO sobreajustan', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Visualización 2D: Árbol de Decisión vs. Random Forest

Observamos cómo Random Forest reemplaza las divisiones rígidas y ortogonales de un solo árbol por una frontera probabilística suave y adaptativa:


In [ ]:
# Comparacion visual en dataset no lineal 2D
X_moons, y_moons = make_moons(n_samples=250, noise=0.30, random_state=42)

tree_single = DecisionTreeClassifier(random_state=42).fit(X_moons, y_moons)
rf_ensemble = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42).fit(X_moons, y_moons)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Arbol individual
DecisionBoundaryDisplay.from_estimator(tree_single, X_moons, cmap='coolwarm', alpha=0.6, ax=axes[0], response_method='predict')
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', edgecolor='k', s=45)
axes[0].set_title(f'Arbol Unico (Acc: {tree_single.score(X_moons, y_moons)*100:.1f}%) - Frontera Rigida', fontweight='bold')

# Random Forest
DecisionBoundaryDisplay.from_estimator(rf_ensemble, X_moons, cmap='coolwarm', alpha=0.6, ax=axes[1], response_method='predict')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', edgecolor='k', s=45)
axes[1].set_title(f'Random Forest (Acc: {rf_ensemble.score(X_moons, y_moons)*100:.1f}%) - Frontera Suavizada', fontweight='bold')

plt.suptitle('Comparativa de Fronteras de Decision: Reduccion de Artefactos de Sobreajuste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Interpretabilidad Global: Importancia de Variables en Random Forest

Al promediar las reducciones de impureza Gini sobre cientos de árboles, la estimación de importancia adquiere mucha mayor estabilidad que en un árbol individual:


In [ ]:
rf_final = RandomForestClassifier(n_estimators=150, max_features='sqrt', random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train)

# MDI
importancias_rf = pd.Series(rf_final.feature_importances_, index=cancer.feature_names).sort_values(ascending=False).head(10)

# Permutacion
perm_rf = permutation_importance(rf_final, X_test, y_test, n_repeats=10, random_state=42)
perm_series = pd.Series(perm_rf.importances_mean, index=cancer.feature_names).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
importancias_rf.plot(kind='barh', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Top 10 Variables - MDI (Random Forest)', fontweight='bold')
axes[0].invert_yaxis()

perm_series.plot(kind='barh', ax=axes[1], color='#2ca02c')
axes[1].set_title('Top 10 Variables - Permutacion en Test', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **Descorrelación Esencial**: La selección aleatoria de variables en cada nodo (`max_features`) reduce la correlación inter-árboles, disminuyendo radicalmente la varianza.
2. **Número de Árboles**: Incrementar `n_estimators` no provoca sobreajuste (a diferencia del número de capas en redes neuronales o rondas de boosting); solo añade estabilidad hasta la saturación.
3. **Algoritmo de Referencia**: Random Forest es uno de los mejores clasificadores *out-of-the-box* para datos tabulares debido a su robustez ante valores atípicos y escasa necesidad de sintonización de hiperparámetros.
